# Init
---

In [1]:
#### mv packages ####
import modules.data as d
import modules.model as m
import modules.pooling as p
import modules.train as t
import modules.utils as u
from pathlib import Path

#### init ####
dataset_dir = Path('/home/mv18gs/Documents/GitHub/pathway_model/datasets/')
device, generator = u.Devices().auto_set_device()#['cuda:1', 'cuda:0'])
# device, generator = u.Devices().set_device('cpu')

#### data ####
brca = d.Preprocessor(
    tcga_project='TCGA-BRCA',
    tcga_dir=dataset_dir/'tcga',
    relation_filepath=dataset_dir/'other'/'relation_ohe.csv',
    metadata_subtype_col = 'paper_BRCA_Subtype_PAM50',
    
    # counts
    apply_DESeq_norm=True, 
    log_transform=True,
    scale_method='standard',

    # etc
    y_col = 'subtype',
    drop = {'subtype':['Normal', 'Primary Tumor']},
    max_subset = 120,
)
_dataset = d.GraphDataset(brca)
_batch = d.get_toy_databatch(_dataset, generator)

# #### Device() ####
# device = cuda:4

# #### Preprocessor() ####
# log0_method              log1p                    str
# class_weights            (6,)                     Tensor (cuda:4)
# edge_index               (2, 32798)               Tensor (cuda:4)
# edge_attr                (32798, 16)              Tensor (cuda:4)
# gene_counts              (4383, 562)              DataFrame
# metadata                 (562, 3)                 DataFrame
# relation                 (32798, 18)              DataFrame
# node_id_map              4383                     dict
# mask_list                305                      list
# mask                     (4383, 305)              Tensor (cuda:4)
# x                        (562, 4383, 1)           Tensor (cuda:4)
# y                        (562,)                   Tensor (cuda:4)
# y_labels                 6                        list
# num_samples              562                      int
# num_nodes                4383                     int


In [2]:
#### convenience variables ####
_embedding_size = 64

# from mask (init)
_mask = brca.mask
_num_nodes, _num_sets = _mask.shape

# from batch (forward)
_batch_size = int(_batch.x.shape[0]/_num_nodes)
_num_node_features = _batch.x.shape[1] # or brca.num_node_features

# Transformer Attention Tests
---
* Input (x) in (batch * nodes, node_features)

* Node-set mask (m) in (nodes, sets)
    * 1 if node is in set, else 0

1. Node to set (masked cross-attention with mask m)
    * GNN: x to node_emb in (batch, nodes, embedding)
    * Query: node_emb to q_set in (batch, sets, embedding)
    * Key: node_emb to k_node in (batch, nodes, embedding)
    * Value: node_emb to v_node in (batch, nodes, embedding)
    * Use m to mask attention; sets only attend to its nodes (indicated by mask)
    * Out: set_emb in (batch, sets, embedding)

2. Set to sample
    * Query: sample (CLS) token to q_sample in (batch, 1, embedding)
    * Key: set_emb to k_set in (batch, sets, embedding)
    * Value: set_emb to v_set in (batch, sets, embedding)
    * Out: samp_emb in (batch, embedding)




In [3]:
import torch
import torch.nn as nn
import torch.nn.functional as F

from modules.model import get_layers, SequentialModel
from modules.pooling import SetPooling
from torch import Tensor
from torch_geometric.nn import MessagePassing, GCNConv
from typing import Literal, Optional, Union

import seaborn as sns


In [4]:
def handle_transformer_dims(embed_dim:Optional[int], head_dim:Optional[int], num_heads:int):
    # none specified; assert error
    assert (embed_dim is not None) or (head_dim is not None), 'one of [embed_dim, head_dim] must be specified'

    # both specified; lin_out reshapes head to embed
    if (embed_dim is not None) and (head_dim is not None):
        return embed_dim, head_dim

    # embed_dim specified; head = embed / num_heads
    elif embed_dim is not None:
        assert embed_dim % num_heads == 0, 'embed_dim must be divisible by num_heads'
        head_dim = embed_dim // num_heads

    # head_dim specified; embed = head * num_heads
    elif head_dim is not None:
        embed_dim = head_dim * num_heads

    return embed_dim, head_dim

In [5]:
class TransformerSetPooling(nn.Module):
    def __init__(
        self,
        mask:Tensor, # (nodes, sets)
        embed_dim:int=None, # provided or = head_dim * heads
        head_dim:int=None, # provided or = embed_dim // heads
        num_heads:int=1,
        dropout:Optional[float]=None,
        *args,
        **kwargs
    ):
        super().__init__(*args, **kwargs)
        self.num_nodes, self.num_sets = mask.shape
        self.mask = mask.T.unsqueeze(0).bool() # (1, num_sets, num_nodes)
        self.num_heads = num_heads
        self.embed_dim, self.head_dim = handle_transformer_dims(embed_dim, head_dim, num_heads)

        self.lin_qkv = nn.Linear(self.embed_dim, 3 * self.num_heads * self.head_dim)
        self.lin_out = nn.Linear(self.num_heads * self.head_dim, self.embed_dim)
        self.dropout = nn.Dropout(p=dropout) if dropout else None
    
    def forward(self, x:Tensor, return_weight:bool=False):
        batch_size = x.shape[0]

        # get qkv
        qkv = self.lin_qkv(x) # (batch, nodes, 3 * heads * head_dim)
        qkv = qkv.reshape(batch_size, self.num_nodes, 3, self.num_heads, self.head_dim)
        qkv = qkv.permute(0,3,1,2,4) # (batch(0), heads(3), nodes(1), qkv(2), head_dim(4))
        q, k, v = qkv[...,0,:], qkv[...,1,:], qkv[...,2,:] # split to (batch, heads, nodes, head_dim) 

        # pool q nodes over sets via mask
        mask = self.mask.expand(batch_size, -1, -1) # broadcast (1, sets, nodes) to (batch, sets, nodes)
        q = torch.einsum('bsn,bhnd->bhsd', mask.float(), q) # mask (batch, heads, nodes, head_dim)

        # compute masked attention weight
        weight = (q @ k.transpose(-2,-1))/(self.head_dim ** 0.5) # bh(sd) @ bh(dn) -> (batch, heads, sets, nodes)
        weight = weight.masked_fill(~mask.unsqueeze(1), float('-inf')) # only allows sets to attend to their nodes
        weight = F.softmax(weight, dim=-1)

        # apply dropout
        if self.dropout is not None:
            weight = self.dropout(weight)

        # compute, format output
        set_emb = weight @ v # bh(sn) @ bh(nd) -> (batch, heads, sets, head_dim)
        set_emb = set_emb.permute(0, 2, 1, 3) # (batch(0), sets(2), heads(1), head_dim(3))
        set_emb = set_emb.reshape(batch_size, self.num_sets, self.num_heads * self.head_dim)

        # project output from heads
        set_emb = self.lin_out(set_emb) # (batch, sets, embed_dim)

        return (set_emb, weight) if return_weight is True else set_emb


In [6]:
# tsp = TransformerSetPooling(
#     mask=_mask,
#     embed_dim=_embedding_size,
#     num_heads=4,
# )
# _x = _batch.x.reshape(_batch_size, _num_nodes, _num_node_features)
# _x = nn.Linear(_num_node_features, _embedding_size)(_x)
# _set_emb = tsp(_x)
# _set_emb.shape

---

In [7]:
class TransformerGlobalPooling(nn.Module):
    def __init__(
        self,
        embed_dim:int=None,
        head_dim:int=None,
        num_heads:int=1,
        dropout:Optional[float]=None,
        *args,
        **kwargs
    ):
        super().__init__(*args, **kwargs)
        self.num_heads = num_heads
        self.embed_dim, self.head_dim = handle_transformer_dims(embed_dim, head_dim, num_heads)
        self.dropout = dropout if dropout is not None else 0.0

        self.attention = nn.MultiheadAttention(self.embed_dim, self.num_heads, dropout=self.dropout, batch_first=True)
        self.cls_token = nn.Parameter(torch.randn(1,1,self.embed_dim)) # (1, 1, embed_dim)
        
    def forward(self, set_emb:Tensor, return_weight:bool=False): # (batch, set, embed_dim)
        batch_size, _, _ = set_emb.shape

        # expand cls to (batch, 1, embed_dim)
        cls = self.cls_token.expand(batch_size, 1, self.embed_dim)

        # MHA, cls (Q) attends to sets (KV)
        samp_emb, weight = self.attention(query=cls, key=set_emb, value=set_emb)
        samp_emb = samp_emb.squeeze(1) # (batch, embed_dim)

        return (samp_emb, weight) if return_weight is True else samp_emb


In [8]:
# tgp = TransformerGlobalPooling(
#     embed_dim=_embedding_size,
#     num_heads=4
# )

# _samp_emb = tgp(_set_emb)
# _samp_emb.shape

## Encoder
---

In [9]:
class Node2SetTransformer(nn.Module):
    def __init__(
        self,
        # required
        num_node_features:int,
        mask:Tensor,
        encoder_class:Union[nn.Module, MessagePassing],

        # transformer
        embed_dim:int=None,
        head_dim:int=None,
        num_heads:int=1,
        dropout:Optional[float]=None,

        # other
        encoder_kwargs:dict={},
        *args,
        **kwargs
    ):
        super().__init__(*args, **kwargs)
        self.num_node_features = num_node_features
        self.num_nodes, self.num_sets = mask.shape
        embed_dim, head_dim = handle_transformer_dims(embed_dim, head_dim, num_heads)

        self.node_encoder = SequentialModel(
            in_channels=self.num_node_features,
            out_channels=embed_dim,
            layer_class=encoder_class,
            **encoder_kwargs
        )

        self.node2set = TransformerSetPooling(
            mask=mask,
            embed_dim=embed_dim,
            head_dim=head_dim,
            num_heads=num_heads,
            dropout=dropout,
        )

        self.set2samp = TransformerGlobalPooling(
            embed_dim=embed_dim,
            head_dim=head_dim,
            num_heads=num_heads,
            dropout=dropout
        )

    def forward(self, x:Tensor, return_set_emb:bool=True, *args, **kwargs):
        # x in (batch * nodes, embed_dim), from PyG DataBatch
        batch_size = int(x.shape[0] / self.num_nodes)
        # x = x.reshape(batch_size, self.num_nodes, self.num_node_features)


        # get node embedding
        x = self.node_encoder(x, *args, **kwargs) # (batch * nodes, embed_dim)
        x = x.view(batch_size, self.num_nodes, -1) # (batch, nodes, embed_dim)

        # node to set embedding
        set_emb = self.node2set(x) # (batch, sets, embed_dim)

        # set to sample embedding
        samp_emb = self.set2samp(set_emb) # (batch, 1, embed_dim)
        samp_emb = samp_emb.squeeze(1) # (batch, embed_dim)

        if return_set_emb:
            return samp_emb, set_emb
        return samp_emb


In [10]:
# n2s = Node2SetTransformer(
#     num_node_features=brca.num_node_features,
#     embed_dim=_embedding_size,
#     mask=brca.mask,
#     encoder_class=GCNConv,
# )

# _samp_emb, _set_emb = n2s(_batch.x, edge_index=_batch.edge_index)

# # sns.heatmap(_samp_emb.detach().cpu())

In [11]:
# sns.heatmap(_set_emb.detach().cpu()[0,:,:])

---

In [12]:
# class FiLMLayer(nn.Module):
#     def __init__(
#         self,
#         in_dim:int,
#         out_dim:int,
#         condition_dim:int,
#         *args,
#         **kwargs
#     ):
#         super().__init__(*args, **kwargs)

#         self.gen = nn.Linear(condition_dim, 2*in_dim)
#         self.lin = nn.Linear(in_dim, out_dim)

#     def forward(self, x, condition):
#         # generate FiLM mod
#         gamma, beta = self.gen(condition).chunk(2, dim=-1)

#         # modulate x
#         x = gamma * x + beta

#         # project
#         x = self.lin(x)
        
#         return x

In [13]:
class Vec2SetUnpooling(nn.Module):
    def __init__(
        self,
        mask:Tensor,

        embed_dim:int=None,
        head_dim:int=None,
        num_heads:int=1,
        dropout:Optional[float]=None,

        film_mod:bool=False,
        transformer_mod:bool=True,
        lin_kwargs:dict={},

        *args,
        **kwargs
    ):
        super().__init__(*args, **kwargs)
        self.mask = mask.T.unsqueeze(0) # (1, sets, nodes)
        _, self.num_sets, self.num_nodes = self.mask.shape
        self.embed_dim, _ = handle_transformer_dims(embed_dim, head_dim, num_heads)
        self.dropout = dropout if dropout is not None else 0.0
        self.film_mod, self.transformer_mod = film_mod, transformer_mod

        self.expand_sample = SequentialModel(
            in_channels = self.embed_dim,
            out_channels = self.embed_dim * self.num_sets,
            layer_class = nn.Linear,
            **lin_kwargs
        )

        
        self.mask_encoder = SequentialModel(
            in_channels = self.num_nodes,
            out_channels = self.embed_dim,
            layer_class = nn.Linear,
            **lin_kwargs
        ) if self.film_mod or self.transformer_mod else None

        self.film_gen = nn.Linear(
            in_features=self.embed_dim,
            out_features=2*self.embed_dim
        ) if self.film_mod else None

        self.attention = nn.MultiheadAttention(
            embed_dim=self.embed_dim,
            num_heads=num_heads,
            dropout=self.dropout,
            batch_first=True
        ) if self.transformer_mod else None

        
    def forward(self, samp_emb:Tensor):
        # samp_emb in (batch, embed_dim)
        batch_size = samp_emb.shape[0]

        # transform sample
        set_emb = self.expand_sample(samp_emb) # (batch, sets * embed_dim)
        set_emb = set_emb.view(batch_size, self.num_sets, self.embed_dim) # (batch, sets, embed_dim)

        # embed mask for context
        if self.film_mod or self.transformer_mod:
            mask_emb = self.mask_encoder(self.mask) # (1, sets, embed_dim)
            mask_emb = mask_emb.expand(batch_size, -1, -1)

        # modulate w/ FiLM
        if self.film_mod:
            gamma, beta = self.film_gen(mask_emb).chunk(2, dim=-1) # (batch, sets, embed_dim)
            set_emb = gamma * set_emb + beta

        # modulate w/ transformer
        if self.transformer_mod:
            set_emb, weight = self.attention(query=mask_emb, key=set_emb, value=set_emb)
            set_emb = set_emb + set_emb

        return set_emb # (batch, sets, embed_dim)


In [14]:
# v2s = Vec2SetUnpooling(
#     mask=brca.mask,
#     embed_dim=_embedding_size,
#     film_mod=True,
#     transformer_mod=True
# )
# _set_recon = v2s(_samp_emb)
# print(_set_recon.shape)

# # sns.heatmap(_set_recon.detach().cpu()[:,0,:])

## Decoder
---

In [15]:
class Set2NodeTransformer(nn.Module):
    def __init__(
        self,
        # required
        # num_node_features:int, 
        mask:Tensor,

        # transformer
        embed_dim:int=None,
        head_dim:int=None,
        num_heads:int=1,
        dropout:Optional[float]=None,

        # etc.
        samp2set_kwargs:Optional[dict]={},
        *args, 
        **kwargs
    ):
        super().__init__(*args, **kwargs)
        self.num_nodes, self.num_sets = mask.shape
        embed_dim, head_dim = handle_transformer_dims(embed_dim, head_dim, num_heads)
        self.dropout = dropout if dropout is not None else 0.0

        self.samp2set = Vec2SetUnpooling(
            mask=mask,
            embed_dim=embed_dim,
            head_dim=head_dim,
            num_heads=num_heads,
            dropout=dropout,
            **samp2set_kwargs
        )

        self.set2node = TransformerSetPooling(
            mask=mask.T,
            embed_dim=embed_dim,
            head_dim=head_dim,
            num_heads=num_heads,
            dropout=dropout
        )

        self.node_decoder = nn.MultiheadAttention(
            embed_dim=embed_dim,
            num_heads=num_heads,
            dropout=self.dropout,
            batch_first=True
        )

    def forward(self, samp_emb:Tensor, return_set_emb:bool=True):
        # reconstruct set from samp_emb
        set_recon = self.samp2set(samp_emb)

        # get node embeddings from set
        node_emb = self.set2node(set_recon)

        # # reconstruct nodes
        node_recon, weight = self.node_decoder(query=node_emb, key=node_emb, value=node_emb)

        return node_recon, set_recon if return_set_emb is True else node_recon



In [16]:
# s2n = SetToNodeTransformer(
#     mask=brca.mask,
#     embed_dim=_embedding_size,
# )

# _node_recon, _set_recon = s2n(_samp_emb)
# print(_node_recon.shape, _set_recon.shape)

## Full model test
---

In [17]:
class PathwayTransformer(nn.Module):
    def __init__(
        self,
        # required
        num_node_features:int,
        mask:Tensor,
        encoder_class:Union[nn.Module, MessagePassing],

        # transformer
        embed_dim:int=None,
        head_dim:int=None,
        num_heads:int=1,
        dropout:Optional[float]=None,

        # other
        encoder_kwargs:dict={},
        samp2set_kwargs:Optional[dict]={},
        *args, **kwargs
    ):
        super().__init__(*args, **kwargs)
        self.num_nodes, self.num_sets = mask.shape
        embed_dim, head_dim = handle_transformer_dims(embed_dim, head_dim, num_heads)

        self.encoder = Node2SetTransformer(
            num_node_features=num_node_features,
            mask=mask,
            encoder_class=encoder_class,
            embed_dim=embed_dim,
            head_dim=head_dim,
            num_heads=num_heads,
            dropout=dropout,
            # encoder_kwargs=encoder_kwargs
        )

        self.decoder = Set2NodeTransformer(
            mask=mask,
            embed_dim=embed_dim,
            head_dim=head_dim,
            num_heads=num_heads,
            dropout=dropout,
            # samp2set_kwargs=samp2set_kwargs            
        )

    def forward(self, x:Tensor, return_set_items:bool=True, *args, **kwargs):
        batch_size = int(x.shape[0] / self.num_nodes)

        z, set_emb = self.encoder(x, *args, **kwargs)

        x_recon, set_recon = self.decoder(z)
        
        # flatten to original shape (batch_size * num_nodes, embedding_size)
        x_recon = x_recon.reshape(batch_size * self.num_nodes, -1)

        return x_recon, set_emb, set_recon if return_set_items else x_recon


In [18]:
# model = PathwayTransformer(
#     num_node_features=brca.num_node_features,
#     mask=brca.mask,
#     encoder_class=nn.Linear,
#     head_dim=8,
# )

# model(_batch.x, edge_index=_batch.edge_index)[0].shape

---

In [19]:
from modules.train import Loader, Trainer

class PathwayTrainer(Trainer):
    def __init__(self, alpha:float=0.5, *args, **kwargs):
        self.alpha = alpha
        super().__init__(*args, **kwargs)
        
    def _compute_loss(self, batch):
        # extract x
        if isinstance(batch, (tuple, list)):
            x = batch[0]
        elif isinstance(batch, dict):
            x = batch.get('x', batch)
        elif hasattr(batch, 'x'):
            x = batch.x
            edge_index = batch.edge_index
        else:
            x = batch

        # forward pass
        x_recon, set_emb, set_recon = self.model(x, edge_index=edge_index)

        # compute reconstruction loss
        loss_x = self.loss_fn(x_recon, x)
        loss_set = self.loss_fn(set_recon, set_emb)
        loss = self.alpha*loss_x + (1-self.alpha)*loss_set

        # tuple outputs
        out = (x_recon, set_recon, set_emb)

        return loss, out

In [21]:
model = PathwayTransformer(
    num_node_features=brca.num_node_features,
    mask=brca.mask,
    encoder_class=nn.Linear,
    head_dim=16,
)

loader = Loader(
    dataset=_dataset,
    generator=generator,
    batch_size=128
)

trainer = PathwayTrainer(
    model=model,
    loader=loader,
    num_epochs=100,
    loss_fn=nn.MSELoss(),
    optimizer_kwargs={'lr':0.001},
    verbose=True,
    alpha=0.5
)

  0%|          | 0/100 [00:00<?, ?it/s]/home/mv18gs/miniconda3/envs/thesis/lib/python3.12/site-packages/torch/utils/_device.py:79: UserWarning: Using a target size (torch.Size([561024, 1])) that is different to the input size (torch.Size([561024, 16])). This will likely lead to incorrect results due to broadcasting. Please ensure they have the same size.
  return func(*args, **kwargs)
/home/mv18gs/miniconda3/envs/thesis/lib/python3.12/site-packages/torch/utils/_device.py:79: UserWarning: Using a target size (torch.Size([43830, 1])) that is different to the input size (torch.Size([43830, 16])). This will likely lead to incorrect results due to broadcasting. Please ensure they have the same size.
  return func(*args, **kwargs)
/home/mv18gs/miniconda3/envs/thesis/lib/python3.12/site-packages/torch/utils/_device.py:79: UserWarning: Using a target size (torch.Size([368172, 1])) that is different to the input size (torch.Size([368172, 16])). This will likely lead to incorrect results due to 

Test	 loss=0.4217

